In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import nibabel as nib
import numpy as np
import shutil
from pathlib import Path
from scipy.ndimage import zoom

MRI_SRC = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_PROCESSED")
MRI_DST = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_STANDARDISED/normalised_mri")
LOCAL   = Path("/content/work_std/mri")
MRI_DST.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

TARGET = (91, 109, 91)
files  = list(MRI_SRC.glob("wr_*.nii"))
print(f"MRI wr_ toplam: {len(files)}")

converted = skipped = error = 0
for i, f in enumerate(files):
    try:
        dst = MRI_DST / f.name
        if dst.exists():
            skipped += 1
            continue

        img  = nib.load(str(f))
        data = img.get_fdata(dtype=np.float32)

        if data.shape != TARGET:
            factors = tuple(t/s for t,s in zip(TARGET, data.shape))
            data = zoom(data, factors, order=1).astype(np.float32)

        local_f = LOCAL / f.name
        nib.save(nib.Nifti1Image(data, img.affine), str(local_f))
        shutil.move(str(local_f), dst)
        converted += 1

        if (i+1) % 200 == 0:
            print(f"  {i+1}/{len(files)} — converted:{converted} skipped:{skipped}")

    except Exception as e:
        error += 1
        print(f"  HATA: {f.name} → {e}")

print(f"\n✅ MRI wr_ bitti — converted:{converted} skipped:{skipped} error:{error}")
print(f"   DST: {len(list(MRI_DST.glob('*.nii')))} / {len(files)}")

MRI wr_ toplam: 3978
  2600/3978 — converted:183 skipped:2417
  2800/3978 — converted:383 skipped:2417
  3000/3978 — converted:583 skipped:2417
  3200/3978 — converted:783 skipped:2417
  3400/3978 — converted:983 skipped:2417
  3600/3978 — converted:1183 skipped:2417
  3800/3978 — converted:1383 skipped:2417

✅ MRI wr_ bitti — converted:1561 skipped:2417 error:0
   DST: 3998 / 3978


In [4]:
import nibabel as nib
from pathlib import Path
from collections import Counter

MRI_STD = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_STANDARDISED/normalised_mri")
PET_WR  = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_PET_SPM_STANDARDISED/normalised_fdg_pet")
PET_SWR = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_PET_SPM_STANDARDISED/smooth_normalised_fdg_pet")

def verify(d, label, expected_count):
    files = list(d.glob("*.nii"))
    shapes = Counter()
    dtypes = Counter()
    for f in files:
        h = nib.load(str(f)).header
        shapes[tuple(h.get_data_shape())] += 1
        dtypes[str(h.get_data_dtype())] += 1

    ok = (
        len(files) == expected_count and
        list(shapes.keys()) == [(91,109,91)] and
        list(dtypes.keys()) == ['float32']
    )

    print(f"\n{'✅' if ok else '❌'} {label}")
    print(f"  Dosya sayısı : {len(files)}  ← {expected_count} olmalı")
    print(f"  Shape        : {dict(shapes)}  ← hepsi (91,109,91) olmalı")
    print(f"  Dtype        : {dict(dtypes)}  ← hepsi float32 olmalı")

verify(MRI_STD, "MRI normalised_mri",            3978)
verify(PET_WR,  "PET normalised_fdg_pet",         1848)
verify(PET_SWR, "PET smooth_normalised_fdg_pet",  1848)


❌ MRI normalised_mri
  Dosya sayısı : 3998  ← 3978 olmalı
  Shape        : {(91, 109, 91): 3998}  ← hepsi (91,109,91) olmalı
  Dtype        : {'float32': 3998}  ← hepsi float32 olmalı

✅ PET normalised_fdg_pet
  Dosya sayısı : 1848  ← 1848 olmalı
  Shape        : {(91, 109, 91): 1848}  ← hepsi (91,109,91) olmalı
  Dtype        : {'float32': 1848}  ← hepsi float32 olmalı

✅ PET smooth_normalised_fdg_pet
  Dosya sayısı : 1848  ← 1848 olmalı
  Shape        : {(91, 109, 91): 1848}  ← hepsi (91,109,91) olmalı
  Dtype        : {'float32': 1848}  ← hepsi float32 olmalı


In [5]:
from pathlib import Path
from collections import Counter

MRI_STD = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_STANDARDISED/normalised_mri")

files = list(MRI_STD.glob("*.nii"))
print(f"Toplam: {len(files)}")

# Tekrar eden subject var mı?
stems = [f.stem for f in files]
counts = Counter(stems)
duplicates = {k:v for k,v in counts.items() if v > 1}
print(f"Tekrar eden dosya: {len(duplicates)}")
for k,v in list(duplicates.items())[:5]:
    print(f"  {k} → {v} kez")

Toplam: 3998
Tekrar eden dosya: 0


In [6]:
from pathlib import Path

MRI_SRC = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_PROCESSED")
MRI_STD = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_STANDARDISED/normalised_mri")

src_names = {f.name for f in MRI_SRC.glob("wr_*.nii")}
std_names = {f.name for f in MRI_STD.glob("*.nii")}

fazla = std_names - src_names
print(f"STD'de olup SRC'de olmayan (fazladan): {len(fazla)}")
for f in sorted(fazla):
    print(f"  {f}")

STD'de olup SRC'de olmayan (fazladan): 20
  wr_I1267018_114_S_2392_T1 (1).nii
  wr_I1294355_035_S_6200_T1 (1).nii
  wr_I1349361_127_S_6512_T1 (1).nii
  wr_I1412023_020_S_6901_T1 (1).nii
  wr_I1507210_024_S_2239_T1 (1).nii
  wr_I1547437_035_S_4464_T1 (1).nii
  wr_I1578905_003_S_6259_T1 (1).nii
  wr_I1615467_135_S_6510_T1 (1).nii
  wr_I199330_072_S_2116_T1 (1).nii
  wr_I208975_031_S_2233_T1 (1).nii
  wr_I223876_068_S_2316_T1 (1).nii
  wr_I232233_130_S_2373_T1 (1).nii
  wr_I241962_005_S_2390_T1 (1).nii
  wr_I250983_029_S_2395_T1 (1).nii
  wr_I256891_136_S_4189_T1 (1).nii
  wr_I257040_099_S_0051_T1 (1).nii
  wr_I264490_137_S_4303_T1 (1).nii
  wr_I270999_003_S_4350_T1 (1).nii
  wr_I280051_114_S_4404_T1 (1).nii
  wr_I285633_073_S_4403_T1 (1).nii


In [7]:
from pathlib import Path

MRI_STD = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_STANDARDISED/normalised_mri")

fazla = [f for f in MRI_STD.glob("*.nii") if "(1)" in f.name]
print(f"Silinecek: {len(fazla)}")
for f in fazla:
    f.unlink()
    print(f"  Silindi: {f.name}")

print(f"\nKalan: {len(list(MRI_STD.glob('*.nii')))}  ← 3978 olmalı")

Silinecek: 20
  Silindi: wr_I1412023_020_S_6901_T1 (1).nii
  Silindi: wr_I256891_136_S_4189_T1 (1).nii
  Silindi: wr_I1615467_135_S_6510_T1 (1).nii
  Silindi: wr_I1294355_035_S_6200_T1 (1).nii
  Silindi: wr_I241962_005_S_2390_T1 (1).nii
  Silindi: wr_I1547437_035_S_4464_T1 (1).nii
  Silindi: wr_I232233_130_S_2373_T1 (1).nii
  Silindi: wr_I1267018_114_S_2392_T1 (1).nii
  Silindi: wr_I280051_114_S_4404_T1 (1).nii
  Silindi: wr_I208975_031_S_2233_T1 (1).nii
  Silindi: wr_I270999_003_S_4350_T1 (1).nii
  Silindi: wr_I257040_099_S_0051_T1 (1).nii
  Silindi: wr_I264490_137_S_4303_T1 (1).nii
  Silindi: wr_I1578905_003_S_6259_T1 (1).nii
  Silindi: wr_I1349361_127_S_6512_T1 (1).nii
  Silindi: wr_I199330_072_S_2116_T1 (1).nii
  Silindi: wr_I285633_073_S_4403_T1 (1).nii
  Silindi: wr_I250983_029_S_2395_T1 (1).nii
  Silindi: wr_I223876_068_S_2316_T1 (1).nii
  Silindi: wr_I1507210_024_S_2239_T1 (1).nii

Kalan: 3978  ← 3978 olmalı


In [9]:
print(f"\nKalan: {len(list(MRI_STD.glob('*.nii')))}  ← 3978 olmalı")


Kalan: 3978  ← 3978 olmalı
